In [ ]:
import torch

torch.cuda.empty_cache()

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def _no_norm_forward_transform(
    self, inputs: torch.Tensor, patched_pads: torch.Tensor
) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
    """Input is of shape [B, N, P]."""
    mu = torch.tensor([0.0], device=inputs.device)
    sigma = torch.tensor([1.0], device=inputs.device)
    outputs = inputs  # no normalization
    return outputs, (mu, sigma)


def _no_norm_reverse_transform(
    self, outputs: torch.Tensor, stats: tuple[torch.Tensor, torch.Tensor]
) -> torch.Tensor:
    return outputs  # no normalization, so reverse is identity

In [ ]:
START_CONTEXT_TIMESTAMP = 1
RESULTS_DIR_BASE_NAME = f"lora-finetuning-{START_CONTEXT_TIMESTAMP}"

In [ ]:
from fusiontimeseries.lib.config import FTSConfig

fts_config = FTSConfig(
    context_length=288,  # divisible by 32 for timesfm patch size
    start_context_timestamp=START_CONTEXT_TIMESTAMP,
    train_context_cutoffs=[1, 70, 139],
    prediction_length=128,
    pred_tail_timestamps=80,
    batch_size=128,
    learning_rate=1e-3,
    lr_scheduler_type="linear",
    optimizer_type="adamw_torch_fused",
    max_grad_norm=1.0,
    max_steps=3000,
    eval_steps=200,
    gradient_accumulation_steps=1,
    full_subsampling=False,
    padding_value=0.0,  # chronos2 has NaN as padding value
    padding_mask_default=0.0,
    padding_mask_indicator=1.0,
)

In [ ]:
from fusiontimeseries.ablations.dataset import BaselineTimeseriesDataset
# from fusiontimeseries.ablations.dataset import FTSAblationIterableDataset

TimeSeriesDataset = BaselineTimeseriesDataset

train_dataset, val_dataset = TimeSeriesDataset.train_val_split(fts_config)

In [ ]:
train_dataset.samples[:4]

In [ ]:
from timesfm import TimesFmCheckpoint, TimesFmHparams
from timesfm.timesfm_torch import TimesFmTorch
from timesfm.pytorch_patched_decoder import PatchedTimeSeriesDecoder


def get_model():

    repo_id = "google/timesfm-2.0-500m-pytorch"

    hparams = TimesFmHparams(
        backend=fts_config.device,  # type: ignore
        per_core_batch_size=fts_config.batch_size,
        horizon_len=fts_config.prediction_length,
        context_len=fts_config.context_length,
        num_layers=50,
        use_positional_embedding=False,
    )

    tfm = TimesFmTorch(
        hparams=hparams, checkpoint=TimesFmCheckpoint(huggingface_repo_id=repo_id)
    )

    model: PatchedTimeSeriesDecoder | None = tfm._model

    if model is None:
        raise ValueError("Model is None")

    return model


model: PatchedTimeSeriesDecoder = get_model()

In [ ]:
# patch the model's normalization functions to no-op for the base ablation
PatchedTimeSeriesDecoder._forward_transform = _no_norm_forward_transform
PatchedTimeSeriesDecoder._reverse_transform = _no_norm_reverse_transform

In [ ]:
# patch config to be HF Trainer compatible

from timesfm.pytorch_patched_decoder import TimesFMConfig
from fusiontimeseries.ablations.trainer import patch_times_fm_config

patch_times_fm_config(TimesFMConfig)

In [ ]:
from fusiontimeseries.loralib.layers import Linear

model = Linear.convert(
    module=model,
    kind="LoRA",
    lora_rank=8,
    lora_alpha=16,
    target_module_names=None,  # slap LoRA on all linear layers
)

In [ ]:
from pathlib import Path

from fusiontimeseries.lib.get_next_path import get_next_path

base_dir = Path("./limitedcontextresults")
base_dir.mkdir(parents=True, exist_ok=True)
output_dir = get_next_path(base_fname=RESULTS_DIR_BASE_NAME, base_dir=base_dir)
output_dir.mkdir(parents=True, exist_ok=False)
print(f"Output directory created at: {output_dir}")

In [ ]:
from fusiontimeseries.loralib.utils import mark_only_lora_as_trainable
from fusiontimeseries.loralib.utils import print_trainable_parameters

mark_only_lora_as_trainable(model=model, bias="none")
print_trainable_parameters(model, save_path=output_dir / "trainable_params.json")

In [ ]:
from transformers import EvalPrediction


def compute_metrics(eval: EvalPrediction) -> dict[str, float]:
    """In HF trainer, line 4911, replace line with logits = outputs  # [1:], otherwise predictions and labels do not have same shape."""
    predictions = eval.predictions
    forecast: torch.Tensor = predictions[:, -1, : fts_config.prediction_length, 0]  # type: ignore
    labels = eval.label_ids
    rmse = ((forecast - labels) ** 2)[-80:].mean() ** 0.5  # type: ignore
    return {"eval_rmse": rmse.item()}

In [ ]:
from transformers.training_args import TrainingArguments

training_arguments = TrainingArguments(
    output_dir=str(output_dir),
    per_device_train_batch_size=fts_config.batch_size,
    per_device_eval_batch_size=fts_config.batch_size,
    learning_rate=fts_config.learning_rate,
    lr_scheduler_type=fts_config.lr_scheduler_type,
    optim=fts_config.optimizer_type,
    logging_strategy="steps",
    logging_steps=fts_config.eval_steps,
    disable_tqdm=False,
    report_to="none",
    max_steps=fts_config.max_steps,
    gradient_accumulation_steps=fts_config.gradient_accumulation_steps,
    dataloader_num_workers=0,
    tf32=False,
    bf16=False,
    save_only_model=True,
    save_total_limit=2,
    save_strategy="steps",
    save_steps=fts_config.eval_steps,
    eval_strategy="steps",
    eval_steps=fts_config.eval_steps,
    load_best_model_at_end=True,  # keep last model since validation set is quite unexpressive
    metric_for_best_model="eval_rmse",
    dataloader_drop_last=False,
    greater_is_better=False,
    use_cpu=False,
    label_names=[
        "future_target"
    ],  # must be truthy for HF Trainer to use overridden compute_loss
    remove_unused_columns=False,  # needed to not accidentally remove columns that our custom compute_loss relies on
    max_grad_norm=fts_config.max_grad_norm,
)
training_arguments._n_gpu = 1

In [ ]:
import json
# from transformers import EarlyStoppingCallback

from fusiontimeseries.ablations.trainer import TimesFMTrainer

trainer = TimesFMTrainer(
    model=model,
    train_args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    fts_config=fts_config,
    compute_metrics=compute_metrics,
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)
with open(output_dir / "training_args.json", "w") as f:
    json.dump(trainer.args.to_dict(), f, indent=4)
fts_config.save_config(output_dir / "fts_config.json")

In [ ]:
import json
from fusiontimeseries.loralib.utils import lora_state_dict

train_output = trainer.train()

with open(output_dir / "train_summary.json", "w") as f:
    json.dump(train_output._asdict(), f, indent=4)

lora_weights = lora_state_dict(model)
torch.save(lora_weights, output_dir / "lora_weights.pt")

In [ ]:
benchmark_data = TimeSeriesDataset.get_benchmark_flux_traces(fts_config)
model = model.eval()

In [ ]:
id_results = TimeSeriesDataset.evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["id"],
    start_context_length=fts_config.start_context_timestamp,
)
round(id_results["rmse"].item(), 4), round(id_results["rmse_standard_error"].item(), 4)

In [ ]:
ood_results = TimeSeriesDataset.evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=benchmark_data["ood"],
    start_context_length=fts_config.start_context_timestamp,
)
(
    round(ood_results["rmse"].item(), 4),
    round(ood_results["rmse_standard_error"].item(), 4),
)

In [ ]:
from fusiontimeseries.lib.dataset import FluxData


flux_data: list[FluxData] = TimeSeriesDataset.load_flux_data(fts_config)
flux_data: list[FluxData] = TimeSeriesDataset.subsample_flux_data(flux_data, 3, 3)

val_flux = {entry.idx: entry for entry in flux_data if entry.is_validation}

train_flux = {entry.idx: entry for entry in flux_data if entry.is_train}

In [ ]:
val_results = TimeSeriesDataset.evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=val_flux,
    start_context_length=fts_config.start_context_timestamp,
)
(
    round(val_results["rmse"].item(), 4),
    round(val_results["rmse_standard_error"].item(), 4),
)

In [ ]:
train_results = TimeSeriesDataset.evaluate_model_on_test_data(
    model=model,
    config=fts_config,
    benchmark_data=train_flux,
    start_context_length=fts_config.start_context_timestamp,
)
(
    round(train_results["rmse"].item(), 4),
    round(train_results["rmse_standard_error"].item(), 4),
)

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

res: dict[str, list[list[float]]]
for IDX in [8, 115, 100, 200, 1, 3]:
    if IDX in [8, 115, 131, 148, 235, 262]:
        res = id_results  # type: ignore type mismatch
        set_name = "ID"
    elif IDX in [0, 1, 2, 3, 4]:
        res = ood_results  # type: ignore type mismatch
        set_name = "OOD"
    elif IDX in [13, 100, 200]:
        res = val_results  # type: ignore type mismatch
        set_name = "VAL"
    else:
        res = train_results  # type: ignore type mismatch
        set_name = "TRAIN"

    gt_mean = np.array(res["ground_truths"][IDX][-80:]).mean()
    fc_mean = np.array(res["forecasts"][IDX][-80:]).mean()

    plt.figure(figsize=(8, 3))
    plt.plot(res["ground_truths"][IDX], label="Ground Truth")
    plt.plot(
        range(fts_config.start_context_timestamp, len(res["forecasts"][IDX])),
        res["forecasts"][IDX][fts_config.start_context_timestamp :],
        label="Forecast",
    )
    plt.axvline(
        x=fts_config.start_context_timestamp,
        color="gray",
        linestyle="--",
        label="Context Cutoff",
    )
    plt.title(
        f"{set_name} ({IDX}) Forecast ({fc_mean:.2f}) vs Ground Truth ({gt_mean:.2f})"
    )
    plt.xlabel("Time Steps")
    plt.ylabel("Flux")
    plt.grid(axis="both", linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / f"{set_name}_{IDX}_forecast.png")
    plt.show()

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

res: dict[str, list[list[float]]]
IDX = 3
if IDX in [8, 115, 131, 148, 235, 262]:
    res = id_results  # type: ignore type mismatch
    set_name = "ID"
elif IDX in [0, 1, 2, 3, 4]:
    res = ood_results  # type: ignore type mismatch
    set_name = "OOD"
elif IDX in [13, 100, 200]:
    res = val_results  # type: ignore type mismatch
    set_name = "VAL"
else:
    res = train_results  # type: ignore type mismatch
    set_name = "TRAIN"

gt_mean = np.array(res["ground_truths"][IDX][-80:]).mean()
fc_mean = np.array(res["forecasts"][IDX][-80:]).mean()

plt.figure(figsize=(8, 3))
plt.plot(res["ground_truths"][IDX], label="Ground Truth")
plt.plot(
    range(fts_config.start_context_timestamp, len(res["forecasts"][IDX])),
    res["forecasts"][IDX][fts_config.start_context_timestamp :],
    label="Forecast",
)
plt.axvline(
    x=fts_config.start_context_timestamp,
    color="gray",
    linestyle="--",
    label="Context Cutoff",
)
plt.title(
    f"{set_name} ({IDX}) Forecast ({fc_mean:.2f}) vs Ground Truth ({gt_mean:.2f})"
)
plt.xlabel("Time Steps")
plt.ylabel("Flux")
plt.grid(axis="both", linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()